# EXP-7 — Metadata Filtering

Restricts retrieval to documents matching a specific metadata value. DirectoryLoader sets the 'source' key automatically. Useful when you have multiple policy files and want queries scoped to one of them.

## Setup

In [18]:
import os
import sys
import time
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

# using the new-era langchain packages, not the old monolith
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings("ignore")

print("all imports loaded")


all imports loaded


In [2]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [3]:
# langsmith traces every chain.invoke() automatically once these env vars are set
# you will see each run in your langsmith project dashboard in real time
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [4]:
# set MLFLOW_TRACKING_URI in your .env to point to a remote mlflow server
# if running locally first: mlflow server --host 0.0.0.0 --port 5000
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("HR_RAG_Experiments")

print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")


mlflow tracking uri: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow


## Load and Prepare Data

In [5]:
# adjust the path and glob pattern to match your folder structure
loader = DirectoryLoader("../data/policies", glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [7]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1329.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [8]:
print(f"eval set: {len(dataset['question'])} question-answer pairs")

eval set: 10 question-answer pairs


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [10]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [11]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm, run_config=RunConfig(max_workers=1))

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from datasets import Dataset, Features, Sequence, Value

def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            answer = chain.invoke(question)
            
            # ensure answer is string
            answers.append(str(answer))

            docs = get_docs_fn(question)
            
            # force pure list of strings (IMPORTANT)
            contexts.append([str(d.page_content) for d in docs])

        except Exception as e:
            print(f"  error on: {question[:60]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    #  FORCE correct schema (THIS FIXES YOUR ERROR)
    features = Features({
        "question": Value("string"),
        "answer": Value("string"),
        "contexts": Sequence(Value("string")),  
        "ground_truth": Value("string"),
    })

    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    }, features=features)

    run_cfg = RunConfig(max_workers=1, timeout=120)

    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,
    )

    return result

In [14]:
def measure_latency(chain, test_q="What is the leave policy?"):
    start = time.time()
    chain.invoke(test_q)
    return round(time.time() - start, 3)


In [15]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
           
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment

In [16]:
# DirectoryLoader sets {"source": "data/policies/filename.md"} on every document
# adjust the filter value below to match an actual file in your data folder

retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"source": "data/policies/leave_policy.md"},
    },
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is the leave policy?"))


I do not know.


In [20]:
result = evaluate_rag(chain, retriever.invoke, dataset)
latency = measure_latency(chain)



Evaluating: 100%|██████████| 30/30 [03:25<00:00,  6.84s/it]


In [21]:
result,latency

({'faithfulness': 0.5000, 'context_precision': 0.0000, 'context_recall': 0.8000},
 0.434)

In [22]:
log_to_mlflow(
    run_name="EXP-7-METADATA",
    result=result,
    latency=latency,
    retriever_type="metadata-filtered",
    top_k=3,
    extra_params={"filter_key": "source", "filter_value": "leave_policy.md"},
)


🏃 View run EXP-7-METADATA at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1/runs/b73bd06099eb46ddafdaf1ab374b2424
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1
